# EHT 2021 uvw comparison (DiFX-correlated FITS-IDI)

`e21d15-1-b1` is an EHT 2021 track (3C273, band 1) correlated with **DiFX**
and exported by `difx2fits` -- a modern counterpart to the 2007 VLBA file in
`Fits_IDI_UVW_Comparison.ipynb`. The compact extract
`tests/unit/data/eht_e21d15_3c273_uvw.npz` (station geometry, source, and
3000 UV rows from job `E21D15.1.bin0000.source0000.FITS`) carries everything
needed. Stations: ALMA (AA), APEX (AX), Greenland (GL), Kitt Peak (KT),
SMT (MG), JCMT (MM), SMA (SW) -- Earth-spanning baselines, mixed alt-az and
Nasmyth mounts (MNTSTA 4/5 are alt-az + Nasmyth focus: identical for uvw).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.time import Time

from radio_telescope_delay_model import calculate_uvw_astropy, calculate_uvw_calc

SPEED_OF_LIGHT = 299792458.0
data = np.load("../tests/unit/data/eht_e21d15_3c273_uvw.npz")
station_name = [str(s) for s in data["station_name"]]
antenna_position = data["antenna_position"]
axis_offset = data["axis_offset"]
print("stations:", station_name)
print("mounts (0 alt-az, 4/5 Nasmyth):", list(data["mntsta"]))
print("source:", data["source_name"], data["source_ra_deg"], data["source_dec_deg"])
antenna1 = (data["BASELINE"] // 256).astype(int)
antenna2 = (data["BASELINE"] % 256).astype(int)
cross = antenna1 != antenna2
file_uvw_all = np.stack([data["UU"], data["VV"], data["WW"]], axis=1) * SPEED_OF_LIGHT
print(
    f"{cross.sum()} cross rows; max |uvw| = {np.abs(file_uvw_all[cross]).max():.0f} m"
)

In [ ]:
rows = np.flatnonzero(cross)
phase_center = np.deg2rad([[data["source_ra_deg"][0], data["source_dec_deg"][0]]])
times = Time(data["DATE"][rows], data["TIME"][rows], format="jd", scale="utc")
unique_unix, inverse = np.unique(times.unix, return_inverse=True)
epochs = Time(unique_unix, format="unix", scale="utc")
print(f"{len(unique_unix)} unique epochs: {epochs[0].isot} -> {epochs[-1].isot}")

uvw_calc, b1, b2 = calculate_uvw_calc(
    antenna_position,
    epochs,
    phase_center,
    reference_position=np.zeros(3),
    station_name=station_name,
    axis_offset_metres=axis_offset,
)
uvw_difx, _, _ = calculate_uvw_calc(
    antenna_position,
    epochs,
    phase_center,
    mode="difxcalc11",
    station_name=station_name,
)
uvw_astropy, _, _ = calculate_uvw_astropy(antenna_position, epochs, phase_center)

pair_index = {(int(x) + 1, int(y) + 1): k for k, (x, y) in enumerate(zip(b1, b2))}
baseline_index, orientation = [], []
for r in rows:
    i, j = antenna1[r], antenna2[r]
    if (i, j) in pair_index:
        baseline_index.append(pair_index[(i, j)])
        orientation.append(1.0)
    else:
        baseline_index.append(pair_index[(j, i)])
        orientation.append(-1.0)
baseline_index = np.array(baseline_index)
orientation = np.array(orientation)[:, None]
file_uvw = file_uvw_all[rows]
models = {
    label: orientation * m[inverse, baseline_index]
    for label, m in (
        ("astropy", uvw_astropy),
        ("calc geometric", uvw_calc),
        ("difxcalc11", uvw_difx),
    )
}

In [ ]:
print(f"{'method':16s} {'matched':>14s} {'flipped':>14s} {'median |res|':>14s}")
for label, model in models.items():
    matched = np.abs(model - file_uvw).max()
    flipped = np.abs(-model - file_uvw).max()
    print(
        f"{label:16s} {matched:12.3f} m {flipped:12.0f} m "
        f"{np.median(np.abs(model - file_uvw)):12.3f} m"
    )

## Interpretation

The flipped column again confirms the archival orientation
`uvw = P(antenna1) - P(antenna2)`. The residual ranking is the **reverse of
the 2007 VLBA file**: here the DiFX-era file's uvw come from the difxcalc11
`.im` model ("ABERRATION CORR: EXACT", aberrated, geocentric), so the
embedded difxcalc11 mode -- bit-identical to that model -- matches best,
limited by the file's float32 storage (~1 m at 1e7 m baselines) and the EOP
values used at correlation time versus today's IERS-B finals. The pure
geometric projection now shows the v/c ~ 1e-4 aberration-convention offset
instead. This is exactly the "more accurate" modern convention: DiFX archives
carry the correlator's aberrated delay-model uvw.

In [ ]:
baseline_length = np.linalg.norm(file_uvw, axis=1)
figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(
    file_uvw[:, 0] / 1e6, file_uvw[:, 1] / 1e6, ".", ms=2, color="k", label="file"
)
axes[0].plot(
    models["difxcalc11"][:, 0] / 1e6,
    models["difxcalc11"][:, 1] / 1e6,
    ".",
    ms=1,
    color="tab:green",
    label="computed (difxcalc11)",
)
axes[0].set_xlabel("u (1000 km)")
axes[0].set_ylabel("v (1000 km)")
axes[0].set_title("EHT uv coverage, 3C273")
axes[0].legend()
axes[0].set_aspect("equal")
for label, color in (
    ("astropy", "tab:orange"),
    ("calc geometric", "tab:blue"),
    ("difxcalc11", "tab:green"),
):
    residual = np.linalg.norm(models[label] - file_uvw, axis=1)
    axes[1].plot(baseline_length / 1e6, residual, ".", ms=2, color=color, label=label)
grid = np.linspace(1e5, baseline_length.max(), 50)
axes[1].plot(grid / 1e6, 1.0e-4 * grid, "k--", lw=1, label="1e-4 x |B| (aberration)")
axes[1].set_yscale("log")
axes[1].set_xlabel("|baseline| (1000 km)")
axes[1].set_ylabel("|uvw residual| (m)")
axes[1].set_title("residual vs baseline")
axes[1].legend(fontsize=8)
plt.tight_layout()